# Daily Challenge — MCP Weather over STDIO

## Complete and heavily documented solution

This notebook builds a local MCP project with no LLM and no external
weather API.

It creates:

- `server.py`, an MCP server named `WeatherDemo`;
- `client.py`, a Python client using STDIO;
- one resource: `cities://list`;
- one tool: `get_weather(city)`;
- an end-to-end terminal capture.

The weather values are intentionally static so the exercise focuses on MCP
discovery and request/response flow.

## Learning objectives

You will practice:

1. MCP host/client/server roles;
2. STDIO child-process transport;
3. the difference between tools and resources;
4. FastMCP decorators and schemas;
5. client initialization and capability discovery;
6. reading a resource;
7. invoking a typed tool;
8. parsing structured tool output;
9. safe logging for STDIO servers;
10. end-to-end validation.

# 1. Understand the architecture

```text
client.py
    │
    ├── starts: mcp run server.py
    │
    ├── sends MCP JSON-RPC through server stdin
    │
    └── receives MCP JSON-RPC through server stdout
```

The client is the local demonstration host. The child process is the MCP
server.

## Resource

`cities://list` is read-only context. It returns the names of cities known by
the demo.

## Tool

`get_weather(city)` is an action. It receives an argument, runs Python
logic, and returns a dictionary.

## Critical STDIO rule

Normal server logs must not be printed to stdout. STDOUT carries MCP
protocol messages. Logging is configured on stderr.

# 2. Install the stable MCP Python SDK

In [ ]:
# The official MCP Python SDK currently has a stable 1.x line.
# `<2` protects this notebook from a future incompatible major release.

%pip install -qU "mcp[cli]>=1.27,<2"

In [ ]:
# Verify Python and MCP versions.

import importlib.metadata as metadata
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("MCP SDK version:", metadata.version("mcp"))

assert sys.version_info >= (3, 10), (
    "Python 3.10 or newer is required."
)

In [ ]:
# Verify that the CLI entry point is available.

!mcp --help | head -n 12

# 3. Create the MCP weather server

`FastMCP` reads Python annotations and docstrings to generate tool schemas.

The server contains:

- a small in-memory weather dictionary;
- a tool with a string input and dictionary output;
- a static resource containing the supported city list;
- stderr logging to make incoming calls visible.

In [ ]:
%%writefile server.py
"""Minimal MCP weather server using the STDIO transport.

The server exposes:
- one read-only resource: cities://list
- one callable tool: get_weather(city)

There are no LLM or external weather API calls. All data is static and kept
in memory so the MCP request/response flow is easy to understand.
"""

import logging
import sys
from typing import Any

from mcp.server.fastmcp import FastMCP


# Configure logging explicitly on stderr.
#
# IMPORTANT:
# With STDIO transport, stdout is reserved for MCP JSON-RPC messages.
# Normal print() calls on stdout can corrupt the protocol connection.
logging.basicConfig(
    level=logging.INFO,
    stream=sys.stderr,
    format="%(levelname)s | WeatherDemo | %(message)s",
)
logger = logging.getLogger("WeatherDemo")


# FastMCP uses Python type hints and docstrings to build MCP schemas.
mcp = FastMCP("WeatherDemo")


# Static in-memory data used by the demonstration.
#
# Keys are normalized lowercase names used for lookup.
# The `city` field preserves the name we want to return to the client.
CITY_DATA: dict[str, dict[str, Any]] = {
    "paris": {
        "city": "Paris",
        "temp_c": 21,
        "condition": "sunny",
    },
    "london": {
        "city": "London",
        "temp_c": 18,
        "condition": "cloudy",
    },
    "nyc": {
        "city": "NYC",
        "temp_c": 24,
        "condition": "breezy",
    },
    # Accept the common full-name alias while returning the canonical NYC
    # record. The resource lists only the three canonical choices.
    "new york": {
        "city": "NYC",
        "temp_c": 24,
        "condition": "breezy",
    },
}

SUPPORTED_CITIES = ("Paris", "London", "NYC")


@mcp.tool()
def get_weather(city: str) -> dict[str, Any]:
    """Return static weather data for one supported city.

    Args:
        city: City name such as Paris, London, NYC, or New York.

    Returns:
        On success, a dictionary containing `city`, `temp_c`, and
        `condition`. If the city is unsupported, an error dictionary is
        returned instead of raising an exception.
    """
    # Normalize the user input so lookup is case-insensitive and tolerant
    # of extra spaces.
    normalized_city = city.strip().lower()

    logger.info(
        "Tool call received: get_weather(city=%r)",
        city,
    )

    weather = CITY_DATA.get(normalized_city)

    if weather is None:
        error_result = {
            "error": f"Unsupported city: {city}",
            "supported_cities": list(SUPPORTED_CITIES),
        }
        logger.info("Returning unsupported-city response")
        return error_result

    # Return a copy so callers cannot mutate the server's source data.
    result = dict(weather)
    logger.info("Returning weather data for %s", result["city"])
    return result


@mcp.resource("cities://list")
def list_cities() -> str:
    """Return supported cities as newline-separated text."""
    logger.info("Resource read received: cities://list")

    # Newline separation makes the resource readable in terminals and UIs.
    return "\n".join(SUPPORTED_CITIES)


def main() -> None:
    """Start the MCP server over standard input/output."""
    logger.info("Starting WeatherDemo over STDIO")

    # Explicitly naming the transport documents the local architecture.
    mcp.run(transport="stdio")


if __name__ == "__main__":
    main()

## Important server decisions

### Input normalization

`city.strip().lower()` allows `Paris`, `PARIS`, and ` paris ` to resolve to
the same record.

### Error dictionaries

An unsupported city returns structured error data rather than crashing the
MCP session.

### Defensive copying

The successful tool result is copied before return, preventing callers from
changing the server's original dictionary.

### Newline-separated resource

`"\n".join(SUPPORTED_CITIES)` produces:

```text
Paris
London
NYC
```

In [ ]:
# Show the generated server file with line numbers.

from pathlib import Path

for number, line in enumerate(
    Path("server.py")
    .read_text(encoding="utf-8")
    .splitlines(),
    start=1,
):
    print(f"{number:>3}: {line}")

In [ ]:
# Compile without starting the infinite STDIO server loop.

import py_compile

py_compile.compile(
    "server.py",
    doraise=True,
)

print("server.py syntax: OK")

# 4. Create the MCP weather client

In [ ]:
%%writefile client.py
"""MCP weather client that starts the local server over STDIO.

The client performs the complete MCP lifecycle:
1. spawn the server with `mcp run`;
2. initialize a ClientSession;
3. discover resources and tools;
4. read cities://list;
5. call get_weather(city="Paris");
6. print and validate the results.
"""

import asyncio
import json
import os
import shutil
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from pydantic import AnyUrl


# Resolve the server path relative to this file rather than the terminal's
# current working directory.
SERVER_PATH = Path(__file__).resolve().with_name(
    "server.py"
)


def find_mcp_command() -> str:
    """Locate the MCP CLI executable installed in the active environment."""
    mcp_command = shutil.which("mcp")

    if mcp_command is None:
        raise RuntimeError(
            "The 'mcp' command was not found. Activate the virtual "
            "environment and install 'mcp[cli]>=1.27,<2'."
        )

    return mcp_command


# This object describes the child process that stdio_client will spawn.
#
# Equivalent shell command:
#     mcp run /absolute/path/to/server.py
server_params = StdioServerParameters(
    command=find_mcp_command(),
    args=["run", str(SERVER_PATH)],
    # Forward the current environment to the server process.
    env=os.environ.copy(),
)


def extract_resource_text(result: Any) -> str:
    """Extract text from an MCP ReadResourceResult."""
    for content_block in getattr(result, "contents", []):
        text = getattr(content_block, "text", None)

        if text is not None:
            return str(text)

    return str(result)


def extract_tool_value(result: Any) -> Any:
    """Extract a Python-friendly value from an MCP CallToolResult.

    FastMCP can expose a typed return value in `structuredContent`, while
    also returning one or more text content blocks. Structured content is
    preferred because the weather tool returns a dictionary.
    """
    structured = getattr(result, "structuredContent", None)

    # Compatibility with SDK objects that expose snake_case attributes.
    if structured is None:
        structured = getattr(result, "structured_content", None)

    if isinstance(structured, dict):
        # FastMCP often wraps a typed function result under "result".
        if set(structured) == {"result"}:
            return structured["result"]

        return structured

    # Fallback for clients/servers that only provide text content.
    for content_block in getattr(result, "content", []):
        if isinstance(content_block, types.TextContent):
            text = content_block.text

            # Try to reconstruct a dictionary if the text is JSON.
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                return text

        text = getattr(content_block, "text", None)

        if text is not None:
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                return text

    return str(result)


async def run() -> None:
    """Connect, discover capabilities, read a resource, and call a tool."""

    # stdio_client starts the child process and yields protocol streams.
    async with stdio_client(server_params) as (
        read_stream,
        write_stream,
    ):
        # ClientSession implements the MCP lifecycle and request methods.
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            # Initialization negotiates protocol versions and capabilities.
            initialization = await session.initialize()

            print(
                "Connected server:",
                initialization.serverInfo.name,
            )

            # Discover concrete resources.
            resources_result = await session.list_resources()
            resource_uris = [
                str(resource.uri)
                for resource in resources_result.resources
            ]

            # Discover callable tools.
            tools_result = await session.list_tools()
            tool_names = [
                tool.name
                for tool in tools_result.tools
            ]

            print("Resources:", resource_uris)
            print("Tools:", tool_names)

            # Read the read-only city resource.
            cities_result = await session.read_resource(
                AnyUrl("cities://list")
            )
            cities_text = extract_resource_text(cities_result)

            print("cities://list ->")
            print(cities_text)

            # Invoke the weather action with JSON-compatible arguments.
            weather_result = await session.call_tool(
                "get_weather",
                arguments={"city": "Paris"},
            )
            weather_value = extract_tool_value(weather_result)

            print(
                "get_weather(Paris) ->",
                weather_value,
            )

            # Smoke-test the end-to-end result.
            assert "cities://list" in resource_uris
            assert "get_weather" in tool_names
            assert cities_text.splitlines() == [
                "Paris",
                "London",
                "NYC",
            ]
            assert isinstance(weather_value, dict)
            assert weather_value["city"] == "Paris"
            assert weather_value["temp_c"] == 21
            assert weather_value["condition"] == "sunny"


def main() -> None:
    """Synchronous program entry point."""
    asyncio.run(run())


if __name__ == "__main__":
    main()

## Client lifecycle

```text
StdioServerParameters
        ↓
stdio_client starts child
        ↓
ClientSession
        ↓
initialize()
        ↓
list_resources() / list_tools()
        ↓
read_resource()
        ↓
call_tool()
```

The client resolves the server path relative to its own file. This avoids a
common failure when the terminal is launched from another folder.

## Why parse `structuredContent`?

The weather tool returns a typed Python dictionary. FastMCP may expose it in
structured form while also returning text content. The client prefers the
structured representation because it preserves keys and numeric values.

In [ ]:
# Show the client file with line numbers.

for number, line in enumerate(
    Path("client.py")
    .read_text(encoding="utf-8")
    .splitlines(),
    start=1,
):
    print(f"{number:>3}: {line}")

In [ ]:
# Static syntax validation for both submission files.

for filename in [
    "server.py",
    "client.py",
]:
    py_compile.compile(
        filename,
        doraise=True,
    )
    print(f"{filename} syntax: OK")

# 5. Run the complete STDIO integration

In [ ]:
# subprocess captures stdout for the required terminal evidence.
# Server logs arrive on stderr because the server uses safe logging.

import subprocess

completed = subprocess.run(
    [sys.executable, "client.py"],
    capture_output=True,
    text=True,
    timeout=30,
    check=False,
)

print("CLIENT STDOUT")
print(completed.stdout)

if completed.stderr.strip():
    print("SERVER/CLIENT STDERR")
    print(completed.stderr)

print("Return code:", completed.returncode)

assert completed.returncode == 0, (
    "The integration failed. Review stderr above."
)

## Expected client output

```text
Connected server: WeatherDemo
Resources: ['cities://list']
Tools: ['get_weather']
cities://list ->
Paris
London
NYC
get_weather(Paris) -> {'city': 'Paris', 'temp_c': 21, 'condition': 'sunny'}
```

The exact representation can vary slightly by SDK version, but the data
must be equivalent.

In [ ]:
# Save a plain-text terminal capture for submission.

terminal_capture = (
    "$ python client.py\n"
    + completed.stdout
)

Path("mcp_weather_terminal_capture.txt").write_text(
    terminal_capture,
    encoding="utf-8",
)

print(terminal_capture)
print("Saved: mcp_weather_terminal_capture.txt")

# 6. Test the unsupported-city behavior

In [ ]:
# Directly import the server module to test its pure Python function.
# This does not open an MCP connection.

import importlib.util

module_spec = importlib.util.spec_from_file_location(
    "weather_server_module",
    Path("server.py").resolve(),
)
weather_server_module = importlib.util.module_from_spec(
    module_spec
)
module_spec.loader.exec_module(
    weather_server_module
)

successful_result = weather_server_module.get_weather(" Paris ")
unsupported_result = weather_server_module.get_weather("Abidjan")

print("Successful direct result:", successful_result)
print("Unsupported direct result:", unsupported_result)

assert successful_result == {
    "city": "Paris",
    "temp_c": 21,
    "condition": "sunny",
}
assert "error" in unsupported_result
assert unsupported_result["supported_cities"] == [
    "Paris",
    "London",
    "NYC",
]

# 7. Run locally

## macOS/Linux

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install "mcp[cli]>=1.27,<2"
python client.py
```

## Windows PowerShell

```powershell
py -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install "mcp[cli]>=1.27,<2"
python client.py
```

The client starts the server automatically.

# 8. Troubleshooting

## `mcp: command not found`

Activate the correct virtual environment and reinstall `mcp[cli]`.

## `Connection closed`

Check for a server syntax/import error:

```bash
python -m py_compile server.py client.py
mcp run server.py
```

Also confirm that server logs are not sent to stdout.

## No resources or tools

Verify the decorators:

```python
@mcp.resource("cities://list")
@mcp.tool()
```

Restart the server after editing.

## Tool validation error

The argument key and type must match the signature:

```python
{"city": "Paris"}
```

## City not found

Use Paris, London, NYC, or New York. Unsupported cities intentionally return
an error dictionary.

# 9. What the data flow looks like

```text
Client request:
tools/call
{
  "name": "get_weather",
  "arguments": {"city": "Paris"}
}

                ↓

Server Python call:
get_weather(city="Paris")

                ↓

Server result:
{
  "city": "Paris",
  "temp_c": 21,
  "condition": "sunny"
}

                ↓

Client prints and validates the dictionary.
```

MCP defines the exchange. No language model is involved.

# Deliverables checklist

- [x] `FastMCP("WeatherDemo")`
- [x] `get_weather(city: str) -> dict`
- [x] Static data for Paris, London, and NYC
- [x] Structured unsupported-city response
- [x] `cities://list` resource
- [x] Newline-separated city names
- [x] STDIO server loop
- [x] Safe stderr logging
- [x] Client starts server through MCP CLI
- [x] Session initialization
- [x] Resource and tool discovery
- [x] Resource read
- [x] Weather tool call
- [x] Structured-result parsing
- [x] Syntax checks
- [x] End-to-end smoke test
- [x] Terminal capture
- [x] Thorough comments and documentation

# Conclusion

This challenge demonstrates a complete MCP exchange without an LLM:

```text
Client
   ↓ STDIO
WeatherDemo server
   ├── Resource: cities://list
   └── Tool: get_weather(city)
```

Discovery lets the client learn what the server offers. Resource and tool
requests then move typed data through a standardized protocol.

# References

- Official MCP Python SDK stable v1:
  https://github.com/modelcontextprotocol/python-sdk/tree/v1.x
- Build an MCP server:
  https://modelcontextprotocol.io/docs/develop/build-server
- MCP architecture:
  https://modelcontextprotocol.io/docs/learn/architecture